In [1]:
# Cell 1 — Imports
import os
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from openai import OpenAI

SUBSETS_DIR = Path("../data/subsets")
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Load API key from environment variable
api_key = os.environ.get("OPENAI_API_KEY")
assert api_key is not None, "OPENAI_API_KEY environment variable not set"
print("API key loaded successfully")

client = OpenAI(api_key=api_key)

API key loaded successfully


In [2]:
# Cell 2 — Load audit scores
audit_df = pd.read_csv(SUBSETS_DIR / "label_audit_scores.csv")
print("Audit scores loaded:", audit_df.shape)
print(audit_df["dataset"].value_counts())
print("\nRecommendation breakdown:")
print(audit_df["recommendation"].value_counts())


Audit scores loaded: (17934, 22)
dataset
isic        9158
chexpert    8776
Name: count, dtype: int64

Recommendation breakdown:
recommendation
keep          11989
downweight     4522
remove         1423
Name: count, dtype: int64


In [3]:
# Cell 3 — Select top suspicious images for explanation
# Top 25 from each dataset = 50 total explanations

TOP_K = 25

top_chexpert = audit_df[
    audit_df["dataset"] == "chexpert"
].sort_values("suspicion_score", ascending=False).head(TOP_K)

top_isic = audit_df[
    audit_df["dataset"] == "isic"
].sort_values("suspicion_score", ascending=False).head(TOP_K)

top_suspicious = pd.concat([top_chexpert, top_isic]).reset_index(drop=True)

print(f"Selected {len(top_suspicious)} images for LLM explanation")
print(top_suspicious["dataset"].value_counts())
print("\nSample of selected images:")
print(top_suspicious[[
    "image_id", "dataset", "label",
    "suspicion_score", "rules_fired"
]].head(10).to_string())

Selected 50 images for LLM explanation
dataset
chexpert    25
isic        25
Name: count, dtype: int64

Sample of selected images:
            image_id   dataset            label  suspicion_score  rules_fired
0  view1_frontal.jpg  chexpert     Lung Opacity         0.961581            0
1  view1_frontal.jpg  chexpert    Pleural Other         0.957289            1
2  view1_frontal.jpg  chexpert         Fracture         0.953832            0
3  view1_frontal.jpg  chexpert      Atelectasis         0.948515            0
4  view1_frontal.jpg  chexpert     Pneumothorax         0.944793            0
5  view1_frontal.jpg  chexpert  Support Devices         0.944679            0
6  view1_frontal.jpg  chexpert      Atelectasis         0.943957            0
7  view1_frontal.jpg  chexpert    Consolidation         0.939931            1
8  view1_frontal.jpg  chexpert            Edema         0.937690            0
9  view1_frontal.jpg  chexpert    Consolidation         0.937424            1


In [4]:
# Cell 4 — Load embeddings for nearest neighbour evidence
embeddings  = np.load(Path("../embeddings/cache/embeddings_resnet18.npy"))
index_df    = pd.read_csv(Path("../embeddings/cache/embedding_index.csv"))

print("Embeddings loaded:", embeddings.shape)
print("Index loaded:     ", index_df.shape)

Embeddings loaded: (17934, 512)
Index loaded:      (17934, 7)


In [ ]:
# Cell 5 — Build nearest neighbour lookup per dataset
# For each flagged image, find its 5 nearest neighbours
# WITHIN its own dataset — used as evidence in the explanation

from sklearn.neighbors import NearestNeighbors

nn_evidence = {}

for dataset_name in ["chexpert", "isic"]:
    ds_mask    = index_df["dataset"].values == dataset_name
    ds_indices = np.where(ds_mask)[0]
    ds_embeddings = embeddings[ds_mask]
    ds_labels     = index_df["label"].values[ds_mask]
    ds_image_ids  = index_df["image_id"].values[ds_mask]

    knn = NearestNeighbors(n_neighbors=6, metric="euclidean", n_jobs=-1)
    knn.fit(ds_embeddings)
    distances, indices = knn.kneighbors(ds_embeddings)

    # For each image in the dataset, store its 5 nearest neighbours for classification
    for local_i, global_i in enumerate(ds_indices):
        image_id = ds_image_ids[local_i]
        # Skip index 0 (the image itself)
        neighbour_local_indices = indices[local_i][1:]
        neighbour_labels  = ds_labels[neighbour_local_indices].tolist()
        neighbour_ids     = ds_image_ids[neighbour_local_indices].tolist()
        neighbour_dists   = distances[local_i][1:].tolist()

        nn_evidence[image_id] = {
            "neighbour_ids"   : neighbour_ids,
            "neighbour_labels": neighbour_labels,
            "neighbour_dists" : [round(d, 3) for d in neighbour_dists]
        }

print("Nearest neighbour evidence built for all images")
print(f"Sample entry:")
sample_id = top_suspicious["image_id"].iloc[0]
print(json.dumps(nn_evidence[sample_id], indent=2))

Nearest neighbour evidence built for all images
Sample entry:
{
  "neighbour_ids": [
    "view1_frontal.jpg",
    "view1_frontal.jpg",
    "view1_frontal.jpg",
    "view1_frontal.jpg",
    "view1_frontal.jpg"
  ],
  "neighbour_labels": [
    "Consolidation",
    "Atelectasis",
    "Lung Opacity",
    "No Finding",
    "Pleural Other"
  ],
  "neighbour_dists": [
    9.859,
    9.886,
    10.192,
    10.283,
    10.323
  ]
}


In [ ]:
# Cell 6 — Build structured evidence prompt per image
# This is the evidence package sent to GPT-4o-mini(OpenAI)

def build_evidence_prompt(row, nn_info):
    dataset    = row["dataset"]
    image_id   = row["image_id"]
    label      = row["label"]
    suspicion  = row["suspicion_score"]
    knn_score  = row["knn_score"]
    mahal      = row["mahal_score"]
    iso        = row["iso_score"]
    rules      = row["rules_fired"]
    uncertain  = row["is_uncertain_original"]
    rec        = row["recommendation"]

    neighbour_labels = nn_info["neighbour_labels"]
    neighbour_dists  = nn_info["neighbour_dists"]

    # Count neighbour label distribution
    from collections import Counter
    neighbour_counts = Counter(neighbour_labels)
    neighbour_summary = ", ".join(
        f"{count} {lbl}" for lbl, count in neighbour_counts.most_common()
    )

    # Build domain context
    if dataset == "chexpert":
        domain_context = (
            "This is a chest X-ray from the CheXpert dataset. "
            "Labels represent thoracic pathologies assigned from "
            "radiology reports using an automated labeller."
        )
        uncertain_note = (
            "NOTE: This image was originally marked as UNCERTAIN (-1) "
            "by the CheXpert labelling process itself."
            if uncertain else ""
        )
    else:
        domain_context = (
            "This is a dermoscopic skin lesion image from the ISIC 2019 "
            "dataset. Labels represent skin condition categories assigned "
            "by dermatologists."
        )
        uncertain_note = ""

    prompt = f"""You are a medical imaging dataset quality auditor. 
Your task is to analyse statistical evidence and generate a clear, 
evidence-driven explanation of why a specific medical image may be 
mislabelled in a training dataset.

DOMAIN CONTEXT:
{domain_context}

IMAGE DETAILS:
- Image ID      : {image_id}
- Assigned Label: {label}
- Dataset       : {dataset.upper()}

STATISTICAL EVIDENCE:
- Suspicion Score    : {suspicion:.3f} (scale 0-1, higher = more suspicious)
- kNN Score          : {knn_score:.3f} (proportion of 15 neighbours with different label)
- Mahalanobis Score  : {mahal:.3f} (distance from class centroid, higher = more outlying)
- Isolation Forest   : {iso:.3f} (anomaly score within class, higher = more anomalous)
- Domain Rules Fired : {int(rules)} out of 4

NEAREST NEIGHBOUR EVIDENCE:
- 5 nearest visual neighbours: {neighbour_summary}
- Distances: {neighbour_dists}
{uncertain_note}

RECOMMENDATION: {rec.upper()}

Based on this evidence, write a concise audit explanation of 3-4 sentences that:
1. States clearly why this image is flagged as suspicious
2. Explains what the visual neighbours suggest about the likely correct label
3. Assesses the confidence level of the suspicion
4. Gives a specific recommendation for a human reviewer

Be direct, technical, and evidence-based. Do not speculate beyond what 
the statistical evidence supports."""

    return prompt


# Test on first image
test_row = top_suspicious.iloc[0]
test_nn  = nn_evidence[test_row["image_id"]]
test_prompt = build_evidence_prompt(test_row, test_nn)
print("Sample prompt built successfully")
print(f"Prompt length: {len(test_prompt)} characters")
print("\n--- SAMPLE PROMPT PREVIEW ---")
print(test_prompt[:500], "...")

Sample prompt built successfully
Prompt length: 1517 characters

--- SAMPLE PROMPT PREVIEW ---
You are a medical imaging dataset quality auditor. 
Your task is to analyse statistical evidence and generate a clear, 
evidence-driven explanation of why a specific medical image may be 
mislabelled in a training dataset.

DOMAIN CONTEXT:
This is a chest X-ray from the CheXpert dataset. Labels represent thoracic pathologies assigned from radiology reports using an automated labeller.

IMAGE DETAILS:
- Image ID      : view1_frontal.jpg
- Assigned Label: Lung Opacity
- Dataset       : CHEXPERT

S ...


In [7]:
# Cell 7 — Call GPT-4o-mini for each flagged image
# Includes rate limiting and error handling

def get_llm_explanation(prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are a medical imaging dataset quality "
                            "auditor. You provide concise, evidence-based "
                            "explanations of potential label errors in "
                            "medical imaging datasets."
                        )
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                max_tokens=300,
                temperature=0.3  # low temperature = more consistent output
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                return f"Explanation generation failed: {str(e)}"


print("Generating LLM explanations for top 50 suspicious images...")
print("This will take approximately 2-3 minutes...\n")

explanations = []

for idx, row in top_suspicious.iterrows():
    image_id = row["image_id"]
    dataset  = row["dataset"]
    label    = row["label"]

    nn_info = nn_evidence.get(image_id, {
        "neighbour_ids"   : [],
        "neighbour_labels": [],
        "neighbour_dists" : []
    })

    prompt      = build_evidence_prompt(row, nn_info)
    explanation = get_llm_explanation(prompt)

    explanations.append({
        "image_id"         : image_id,
        "dataset"          : dataset,
        "label"            : label,
        "suspicion_score"  : row["suspicion_score"],
        "knn_score"        : row["knn_score"],
        "mahal_score"      : row["mahal_score"],
        "iso_score"        : row["iso_score"],
        "rules_fired"      : row["rules_fired"],
        "recommendation"   : row["recommendation"],
        "is_uncertain_original": row["is_uncertain_original"],
        "neighbour_labels" : str(nn_info["neighbour_labels"]),
        "explanation"      : explanation
    })

    print(f"[{idx+1:02d}/50] {dataset:8} | {label:35} | "
          f"suspicion={row['suspicion_score']:.3f} ✓")

    # Small delay to avoid rate limiting
    time.sleep(0.5)

explanations_df = pd.DataFrame(explanations)
print(f"\nDone. Generated {len(explanations_df)} explanations.")

Generating LLM explanations for top 50 suspicious images...
This will take approximately 2-3 minutes...

[01/50] chexpert | Lung Opacity                        | suspicion=0.962 ✓
[02/50] chexpert | Pleural Other                       | suspicion=0.957 ✓
[03/50] chexpert | Fracture                            | suspicion=0.954 ✓
[04/50] chexpert | Atelectasis                         | suspicion=0.949 ✓
[05/50] chexpert | Pneumothorax                        | suspicion=0.945 ✓
[06/50] chexpert | Support Devices                     | suspicion=0.945 ✓
[07/50] chexpert | Atelectasis                         | suspicion=0.944 ✓
[08/50] chexpert | Consolidation                       | suspicion=0.940 ✓
[09/50] chexpert | Edema                               | suspicion=0.938 ✓
[10/50] chexpert | Consolidation                       | suspicion=0.937 ✓
[11/50] chexpert | Pleural Effusion                    | suspicion=0.936 ✓
[12/50] chexpert | Support Devices                     | suspicion=0.9

In [8]:
# Cell 8 — Preview explanations
print("SAMPLE EXPLANATIONS\n")
print("=" * 70)

for _, row in explanations_df.head(4).iterrows():
    print(f"\nImage ID  : {row['image_id']}")
    print(f"Dataset   : {row['dataset'].upper()}")
    print(f"Label     : {row['label']}")
    print(f"Suspicion : {row['suspicion_score']:.3f}")
    print(f"Action    : {row['recommendation'].upper()}")
    print(f"\nExplanation:")
    print(row["explanation"])
    print("-" * 70)

SAMPLE EXPLANATIONS


Image ID  : view1_frontal.jpg
Dataset   : CHEXPERT
Label     : Lung Opacity
Suspicion : 0.962
Action    : REMOVE

Explanation:
The image "view1_frontal.jpg" is flagged as suspicious primarily due to its high suspicion score of 0.962, indicating a strong likelihood of misclassification. The nearest visual neighbours include a mix of pathologies, with only one neighbour also labeled as Lung Opacity, suggesting potential misalignment in the assigned label. The Mahalanobis score of 14.217 indicates that this image is significantly outlying compared to the class centroid for Lung Opacity, further supporting the suspicion of mislabeling. Given these factors, I recommend that a human reviewer closely examine this image to verify its correct classification, as the statistical evidence suggests it may not accurately represent Lung Opacity.
----------------------------------------------------------------------

Image ID  : view1_frontal.jpg
Dataset   : CHEXPERT
Label     : 

In [9]:
# Cell 9 — Save explanations CSV
explanations_df.to_csv(
    REPORTS_DIR / "llm_explanations.csv", index=False
)
print(f"Saved llm_explanations.csv ({len(explanations_df)} rows)")

Saved llm_explanations.csv (50 rows)


In [ ]:
# Cell 10 — Generating HTML audit report
def generate_html_report(explanations_df, audit_df):

    total_audited  = len(audit_df)
    total_removed  = (audit_df["recommendation"] == "remove").sum()
    total_downwt   = (audit_df["recommendation"] == "downweight").sum()
    chex_total     = (audit_df["dataset"] == "chexpert").sum()
    isic_total     = (audit_df["dataset"] == "isic").sum()
    chex_removed   = (
        (audit_df["dataset"] == "chexpert") &
        (audit_df["recommendation"] == "remove")
    ).sum()
    isic_removed   = (
        (audit_df["dataset"] == "isic") &
        (audit_df["recommendation"] == "remove")
    ).sum()

    # Build image cards HTML
    cards_html = ""
    for _, row in explanations_df.iterrows():
        suspicion_pct = int(row["suspicion_score"] * 100)

        if row["recommendation"] == "remove":
            badge_color = "#dc3545"
            badge_text  = "REMOVE"
        elif row["recommendation"] == "downweight":
            badge_color = "#fd7e14"
            badge_text  = "DOWNWEIGHT"
        else:
            badge_color = "#28a745"
            badge_text  = "KEEP"

        cards_html += f"""
        <div class="card">
            <div class="card-header">
                <span class="image-id">{row['image_id']}</span>
                <span class="badge" 
                      style="background:{badge_color}">{badge_text}</span>
            </div>
            <div class="card-meta">
                <span class="tag">{row['dataset'].upper()}</span>
                <span class="tag">{row['label']}</span>
                <span class="tag">Suspicion: {row['suspicion_score']:.3f}</span>
                <span class="tag">Rules fired: {int(row['rules_fired'])}</span>
            </div>
            <div class="scores">
                <div class="score-item">
                    <span class="score-label">kNN</span>
                    <div class="score-bar">
                        <div class="score-fill" 
                             style="width:{int(row['knn_score']*100)}%;
                                    background:#6c757d"></div>
                    </div>
                    <span class="score-val">{row['knn_score']:.3f}</span>
                </div>
                <div class="score-item">
                    <span class="score-label">Suspicion</span>
                    <div class="score-bar">
                        <div class="score-fill" 
                             style="width:{suspicion_pct}%;
                                    background:#dc3545"></div>
                    </div>
                    <span class="score-val">{row['suspicion_score']:.3f}</span>
                </div>
            </div>
            <div class="neighbours">
                <strong>Nearest neighbours:</strong> 
                {row['neighbour_labels']}
            </div>
            <div class="explanation">
                <strong>Audit Explanation:</strong><br>
                {row['explanation']}
            </div>
        </div>
        """

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Medical Label Audit Report</title>
    <style>
        body {{
            font-family: -apple-system, BlinkMacSystemFont,
                         'Segoe UI', sans-serif;
            background: #f8f9fa;
            color: #212529;
            margin: 0;
            padding: 20px;
        }}
        .header {{
            background: #1a1a2e;
            color: white;
            padding: 30px 40px;
            border-radius: 12px;
            margin-bottom: 30px;
        }}
        .header h1 {{ margin: 0 0 8px 0; font-size: 1.8em; }}
        .header p  {{ margin: 0; opacity: 0.7; }}
        .summary {{
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            gap: 16px;
            margin-bottom: 30px;
        }}
        .stat-box {{
            background: white;
            border-radius: 10px;
            padding: 20px;
            text-align: center;
            box-shadow: 0 2px 8px rgba(0,0,0,0.08);
        }}
        .stat-box .number {{
            font-size: 2em;
            font-weight: 700;
            color: #1a1a2e;
        }}
        .stat-box .label {{
            font-size: 0.85em;
            color: #6c757d;
            margin-top: 4px;
        }}
        .section-title {{
            font-size: 1.3em;
            font-weight: 600;
            margin: 30px 0 16px 0;
            padding-bottom: 8px;
            border-bottom: 2px solid #dee2e6;
        }}
        .card {{
            background: white;
            border-radius: 10px;
            padding: 20px 24px;
            margin-bottom: 20px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.08);
            border-left: 4px solid #dc3545;
        }}
        .card-header {{
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 10px;
        }}
        .image-id {{
            font-family: monospace;
            font-size: 0.95em;
            color: #495057;
        }}
        .badge {{
            color: white;
            padding: 3px 10px;
            border-radius: 20px;
            font-size: 0.78em;
            font-weight: 600;
        }}
        .card-meta {{
            display: flex;
            gap: 8px;
            flex-wrap: wrap;
            margin-bottom: 12px;
        }}
        .tag {{
            background: #f1f3f5;
            padding: 2px 10px;
            border-radius: 20px;
            font-size: 0.8em;
            color: #495057;
        }}
        .scores {{ margin-bottom: 12px; }}
        .score-item {{
            display: flex;
            align-items: center;
            gap: 10px;
            margin-bottom: 6px;
        }}
        .score-label {{
            width: 70px;
            font-size: 0.8em;
            color: #6c757d;
        }}
        .score-bar {{
            flex: 1;
            background: #e9ecef;
            border-radius: 4px;
            height: 8px;
            overflow: hidden;
        }}
        .score-fill {{
            height: 100%;
            border-radius: 4px;
        }}
        .score-val {{
            width: 40px;
            font-size: 0.8em;
            color: #495057;
            text-align: right;
        }}
        .neighbours {{
            font-size: 0.85em;
            color: #6c757d;
            margin-bottom: 10px;
            padding: 8px 12px;
            background: #f8f9fa;
            border-radius: 6px;
        }}
        .explanation {{
            font-size: 0.9em;
            line-height: 1.6;
            color: #343a40;
            padding: 12px;
            background: #fff9f0;
            border-radius: 6px;
            border-left: 3px solid #fd7e14;
        }}
    </style>
</head>
<body>
    <div class="header">
        <h1>Medical Imaging Label Audit Report</h1>
        <p>Agentic AI System for Automated Label Auditing — 
           MSc Dissertation Project</p>
    </div>

    <div class="summary">
        <div class="stat-box">
            <div class="number">{total_audited}</div>
            <div class="label">Total Images Audited</div>
        </div>
        <div class="stat-box">
            <div class="number" style="color:#dc3545">
                {total_removed}
            </div>
            <div class="label">Recommended for Removal</div>
        </div>
        <div class="stat-box">
            <div class="number" style="color:#fd7e14">
                {total_downwt}
            </div>
            <div class="label">Recommended for Downweighting</div>
        </div>
        <div class="stat-box">
            <div class="number" 
                 style="color:#28a745">
                {total_audited - total_removed - total_downwt}
            </div>
            <div class="label">Kept at Full Confidence</div>
        </div>
    </div>

    <div class="summary">
        <div class="stat-box">
            <div class="number">{chex_total}</div>
            <div class="label">CheXpert Images</div>
        </div>
        <div class="stat-box">
            <div class="number" style="color:#dc3545">
                {chex_removed}
            </div>
            <div class="label">CheXpert Removals</div>
        </div>
        <div class="stat-box">
            <div class="number">{isic_total}</div>
            <div class="label">ISIC Images</div>
        </div>
        <div class="stat-box">
            <div class="number" style="color:#dc3545">
                {isic_removed}
            </div>
            <div class="label">ISIC Removals</div>
        </div>
    </div>

    <div class="section-title">
        Top 50 Most Suspicious Images — Audit Findings
    </div>

    {cards_html}

</body>
</html>"""

    return html


html_report = generate_html_report(explanations_df, audit_df)

report_path = REPORTS_DIR / "audit_report.html"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(html_report)

print(f"HTML audit report saved: {report_path}")
print(f"Open this file in your browser to view the report")

HTML audit report saved: ..\reports\audit_report.html
Open this file in your browser to view the report


In [11]:
# Cell 11 — Final verification
print("NOTEBOOK 09 — FINAL VERIFICATION\n")

explanations = pd.read_csv(REPORTS_DIR / "llm_explanations.csv")
print(f"llm_explanations.csv : {len(explanations)} rows")
print(f"audit_report.html    : "
      f"{(REPORTS_DIR / 'audit_report.html').stat().st_size / 1024:.1f} KB")

print("\nExplanation coverage:")
print(explanations["dataset"].value_counts().to_string())

print("\nRecommendation breakdown in explanations:")
print(explanations["recommendation"].value_counts().to_string())

print("\nSample explanation (first CheXpert entry):")
chex_sample = explanations[
    explanations["dataset"] == "chexpert"
].iloc[0]
print(f"  Image : {chex_sample['image_id']}")
print(f"  Label : {chex_sample['label']}")
print(f"  Score : {chex_sample['suspicion_score']:.3f}")
print(f"  Text  : {chex_sample['explanation'][:200]}...")

print("\nAll outputs saved to ../reports/")
print("Notebook 09 complete.")

NOTEBOOK 09 — FINAL VERIFICATION

llm_explanations.csv : 50 rows
audit_report.html    : 126.5 KB

Explanation coverage:
dataset
chexpert    25
isic        25

Recommendation breakdown in explanations:
recommendation
remove    50

Sample explanation (first CheXpert entry):
  Image : view1_frontal.jpg
  Label : Lung Opacity
  Score : 0.962
  Text  : The image "view1_frontal.jpg" is flagged as suspicious primarily due to its high suspicion score of 0.962, indicating a strong likelihood of misclassification. The nearest visual neighbours include a ...

All outputs saved to ../reports/
Notebook 09 complete.
